In [1]:
from qiskit.dagcircuit import DAGOpNode

def assign_moments_topo(dag):
    """Moment(n) = 0 if no predecessors else 1 + max Moment(pred)."""
    moments = {}
    for node in dag.topological_op_nodes():   # stable identity
        preds = [e.node for e in dag.predecessors(node)
                 if hasattr(e, "node") and isinstance(e.node, DAGOpNode)]
        moments[node] = 0 if not preds else 1 + max(moments[p] for p in preds)
    return moments


In [2]:
import importlib, lightcone
# patch lightcone
lightcone.assign_moments = assign_moments_topo

import circuits_to_graph
# reload so its "from lightcone import assign_moments" picks up the patched one
importlib.reload(circuits_to_graph)

import dataset
# reload so its "from circuits_to_graph import circuit_to_gategraph_data" rebinds to the reloaded module
importlib.reload(dataset)


<module 'dataset' from '/home/macula/SMATousi/projects/quantum/vigir-ml-qem/docs/demos/graph_approach/dataset.py'>

In [3]:
# If needed (comment out if already installed):
# !pip install qiskit torch torch-geometric networkx

import os, json, torch
from pathlib import Path

# Make the local package files importable
import sys
sys.path.append(str(Path(".").resolve()))

from schemas import ConvertConfig
from dataset import convert_folder_to_pt


In [4]:
INPUT_DIR = "../../tutorials/data/ising_zne_hardware/100q_brisbane/"   # <- EDIT ME
OUT_PT    = "./processed/graphs_train.pt"          # where to save the collated graphs
MEASURED  = [0, 1, 2, 3, 4]            # <- EDIT if needed

cfg = ConvertConfig(
    input_dir=INPUT_DIR,
    split="train",
    output_path=OUT_PT,
    measured_qubits=MEASURED,
    measured_map_json=None,          # or "measured_map.json"
    max_params=2,
    verbose=True,
    file_extensions=[".qpy",".qasm",".pk",".pickle"],
)
cfg


ConvertConfig(input_dir='../../tutorials/data/ising_zne_hardware/100q_brisbane/', split='train', output_path='./processed/graphs_train.pt', measured_qubits=[0, 1, 2, 3, 4], measured_map_json=None, max_params=2, max_qubits=None, verbose=True, file_extensions=['.qpy', '.qasm', '.pk', '.pickle'])

In [5]:
out_path = convert_folder_to_pt(cfg)
print("Saved:", out_path)


Processing...


Converting train circuits:   0%|          | 0/500 [00:00<?, ?circuit/s]

Saved: ./processed/graphs_train.pt


Done!


In [7]:
import torch
from torch_geometric.data import Data

graphs = torch.load(OUT_PT, weights_only=False)   # this is a Python list of Data
print(f"Loaded {len(graphs)} graphs")

g0 = graphs[0]
print("x:", g0.x.shape, "edge_index:", g0.edge_index.shape, "LC:", g0.lightcone_masks.shape)

def sanity_report(g: Data):
    n = g.x.size(0)
    ei = g.edge_index
    if ei.numel():
        # drop any edges that are out of range, just in case
        valid = (ei[0] >= 0) & (ei[1] >= 0) & (ei[0] < n) & (ei[1] < n)
        if not valid.all():
            # repair in place so downstream never breaks
            g.edge_index = ei[:, valid]
        ei = g.edge_index
        assert int(ei.max()) < n and int(ei.min()) >= 0, "edge_index out of bounds"
    # check LC dims: (num_nodes, num_measured)
    if g.lightcone_masks.numel():
        assert g.lightcone_masks.shape[0] == n, "LC row count != num_nodes"
    return n, ei.size(1)

total_n = total_e = 0
bad = 0
for i, gi in enumerate(graphs):
    try:
        n, e = sanity_report(gi)
        total_n += n; total_e += e
    except AssertionError as err:
        bad += 1
        print(f"Graph {i} failed: {err}")

print(f"OK ✓  graphs={len(graphs)} | avg nodes={total_n/len(graphs):.1f} | avg edges={total_e/len(graphs):.1f} | repaired={bad}")


Loaded 500 graphs
x: torch.Size([2071, 20]) edge_index: torch.Size([2, 2565]) LC: torch.Size([2071, 5])
OK ✓  graphs=500 | avg nodes=11101.4 | avg edges=13822.9 | repaired=0
